# AMT Benchmark Runner

Use this notebook to launch short `amg.py` runs, capture their console metrics, and visualize how the current benchmark behaves without editing the training script.

The suggested workflow is:
1. Load environment defaults (optionally reading `.env`).
2. Tweak the CLI overrides dictionary to reflect the scenario you want to test.
3. Call `run_benchmark` to stream logs + persist them under `logs/`.
4. Convert the captured metrics into a DataFrame/plot for quick inspection.

> Tip: keep `total_steps` divisible by `num_envs × horizon` so PPO completes whole updates.


In [1]:
import datetime as dt
import os
import re
import shlex
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
PYTHON = sys.executable

def load_env_file(path: Path = REPO_ROOT / ".env") -> None:
    'Best-effort loader so W&B/auth tokens can be picked up without extra steps.'
    if not path.exists():
        print("No .env file detected; using existing shell environment.")
        return
    loaded = 0
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())
        loaded += 1
    print(f"Loaded {loaded} entries from {path}")

load_env_file()
print(f"Python executable: {PYTHON}")
print(f"Repository root : {REPO_ROOT}")


Loaded 2 entries from /raid/nobackup/trustvision/hungbui/workspace/per/amt/.env
Python executable: /raid/nobackup/trustvision/hungbui/workspace/per/amt/.venv/bin/python
Repository root : /raid/nobackup/trustvision/hungbui/workspace/per/amt


## Configure CLI defaults

Edit `BASE_ARGS` / `FLAG_DEFAULTS` to reflect your usual benchmark settings (device, AMP, logging, etc.). Overrides supplied later are merged on top of these defaults.

If you need extra packages for analysis (e.g. pandas/matplotlib), run `%pip install pandas matplotlib seaborn` in a new cell before plotting.


In [2]:
BASE_ARGS = {
    "--env-id": "CartPole-v1",
    "--device": "cpu",
    "--num-envs": 4,
    "--horizon": 128,
    "--total-steps": 32768,
    "--log-interval": 5,
    "--seed": 1,
}

FLAG_DEFAULTS = {
    "--wandb": False,
    "--amp": False,
}

def _validate_schedule(args: dict) -> None:
    batches = int(args.get("--num-envs", 1)) * int(args.get("--horizon", 1))
    total = int(args.get("--total-steps", batches))
    if total % batches != 0:
        raise ValueError("total_steps must be divisible by num_envs × horizon")

def build_command(overrides: dict | None = None) -> tuple[list[str], dict]:
    merged: dict = {**BASE_ARGS, **FLAG_DEFAULTS}
    if overrides:
        merged.update(overrides)
    _validate_schedule(merged)
    cmd: list[str] = [PYTHON, "amg.py"]
    resolved: dict = {}
    for key, value in merged.items():
        if isinstance(value, bool):
            resolved[key] = value
            if value:
                cmd.append(key)
            continue
        if value is None:
            continue
        resolved[key] = value
        cmd.extend([key, str(value)])
    return cmd, resolved

def format_command(cmd: list[str]) -> str:
    return " ".join(shlex.quote(part) for part in cmd)

example_cmd, _ = build_command()
print("Default command:", format_command(example_cmd))


Default command: /raid/nobackup/trustvision/hungbui/workspace/per/amt/.venv/bin/python amg.py --env-id CartPole-v1 --device cpu --num-envs 4 --horizon 128 --total-steps 32768 --log-interval 5 --seed 1


## Launch + parse helper

`run_benchmark` streams `amg.py` output, stores the raw log under `logs/`, and extracts the PPO summary lines so you can build tables/plots later. Interrupt with `Kernel ▸ Interrupt` to stop early.


In [3]:
LOG_PATTERN = re.compile(
    r"update=(?P<update>\d+)\s+episodes=(?P<episodes>\d+)\s+ret50=(?P<ret50>-?\d+\.?\d*)\s+len50=(?P<len50>-?\d+\.?\d*)\s+gate_mean=(?P<gate>-?\d+\.?\d*)\s+kl=(?P<kl>-?\d+\.?\d*)\s+clipfrac=(?P<clip>-?\d+\.?\d*)"
)

def run_benchmark(overrides: dict | None = None, *, log_name: str | None = None) -> dict:
    cmd, args = build_command(overrides)
    log_dir = REPO_ROOT / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"{log_name or 'benchmark'}_{stamp}.log"
    print("Launching:", format_command(cmd))
    metrics: list[dict] = []
    proc: subprocess.Popen[str] | None = None
    try:
        with open(log_path, "w", encoding="utf-8") as sink:
            proc = subprocess.Popen(
                cmd,
                cwd=REPO_ROOT,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            assert proc.stdout is not None
            num_envs = int(args.get("--num-envs", 1))
            horizon = int(args.get("--horizon", 1))
            frames_per_update = num_envs * horizon
            try:
                for line in proc.stdout:
                    print(line, end="")
                    sink.write(line)
                    sink.flush()
                    match = LOG_PATTERN.search(line)
                    if match:
                        update = int(match.group("update"))
                        metrics.append(
                            {
                                "update": update,
                                "episodes": int(match.group("episodes")),
                                "ret50": float(match.group("ret50")),
                                "len50": float(match.group("len50")),
                                "gate_mean": float(match.group("gate")),
                                "kl": float(match.group("kl")),
                                "clipfrac": float(match.group("clip")),
                                "frames": frames_per_update * update,
                            }
                        )
            except KeyboardInterrupt:
                print("Interrupted — terminating run...")
                proc.terminate()
                raise
            finally:
                proc.stdout.close()
        return_code = proc.wait()
    finally:
        if proc and proc.poll() is None:
            proc.kill()
    if return_code != 0:
        raise RuntimeError(f"amg.py exited with status {return_code}")
    print(f"Log saved to {log_path}")
    return {"cmd": cmd, "args": args, "log_path": log_path, "metrics": metrics}


## Run a short benchmark

Adjust the overrides below (longer horizons, CUDA devices, AMP, etc.) and execute the cell to kick off a run. The defaults keep things quick on CPU.


In [4]:
benchmark_overrides = {
    "--total-steps": 16384,
    "--num-envs": 4,
    "--log-interval": 2,
    # Example toggles:
    # "--device": "cuda",
    # "--amp": True,
    # "--wandb": True,
}

last_run = None
if benchmark_overrides is not None:
    last_run = run_benchmark(benchmark_overrides)


Launching: /raid/nobackup/trustvision/hungbui/workspace/per/amt/.venv/bin/python amg.py --env-id CartPole-v1 --device cpu --num-envs 4 --horizon 128 --total-steps 16384 --log-interval 2 --seed 1
Interrupted — terminating run...


KeyboardInterrupt: 

## Build a DataFrame (requires pandas)

Run this cell after `last_run` is populated to get a quick tabular view of the PPO statistics.


In [ ]:
import importlib

def run_to_dataframe(run_result: dict):
    pandas_spec = importlib.util.find_spec("pandas")
    if pandas_spec is None:
        raise RuntimeError("Install pandas via `%pip install pandas` to enable this cell.")
    import pandas as pd  # type: ignore

    df = pd.DataFrame(run_result.get("metrics", []))
    return df

if 'last_run' in globals() and last_run is not None:
    last_df = run_to_dataframe(last_run)
    display(last_df.tail())
else:
    print("No run captured yet — execute the benchmark cell first.")


## Plot reward trends (optional)

This cell uses matplotlib; install it first if it is missing. Modify the plotting logic to overlay multiple experiments or additional metrics.


In [ ]:
if 'last_df' in globals() and not last_df.empty:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(last_df['frames'], last_df['ret50'], label='ret50 (rolling return)')
    ax.set_xlabel('Frames')
    ax.set_ylabel('Return')
    ax.set_title('AMT benchmark trend')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()
else:
    print("No metrics to plot yet.")
